# Разбор Физически Горячего `O/B-like` Подмножества Внутри True `O`

Цель:
- Выделить внутри true `O` только физически горячее подмножество по baseline `teff_gspphot >= 10000 K`.
- Проверить, исчезает ли после этого cool contamination из полного `O` source.
- Понять, остается ли проблема уже как узкая граница `O -> B`, а не как broad failure всего coarse-layer.
- Зафиксировать evidence до любого retrain, rebalance или специальной policy для `O`.

In [1]:
# Setup: repo root, sys.path и базовые display-настройки.
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import seaborn as sns
from IPython.display import display

def find_repo_root(start_path: Path) -> Path:
    current_path = start_path.resolve()
    while current_path != current_path.parent:
        if (current_path / 'src').exists() and (current_path / '.git').exists():
            return current_path
        current_path = current_path.parent
    raise RuntimeError('Could not locate repository root.')

REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 140)

In [2]:
# Импортируем review helpers после добавления src в sys.path.
from exohost.reporting.archive_research.coarse_o_hot_subset_review import (
    HotOBLikeSubsetConfig,
    load_hot_ob_like_subset_review_bundle_from_env,
    build_hot_subset_membership_summary_frame,
    build_hot_subset_quality_summary_frame,
    build_hot_subset_quality_reason_frame,
    build_hot_subset_scored_prediction_frame,
    build_hot_subset_predicted_physics_summary_frame,
    build_hot_subset_final_outcome_distribution_frame,
    build_hot_subset_final_coarse_distribution_frame,
    build_hot_subset_final_reason_frame,
    build_hot_subset_high_confidence_non_ob_preview_frame,
)

## План

1. Выделяем внутри true `O` hot-subset по baseline `teff_gspphot >= 10000 K`.
2. Смотрим, сколько строк остается после этого physics-cut и сколько из них проходят `quality_state = pass`.
3. Считаем coarse predictions только на hot pass-subset.
4. Проверяем, исчезают ли `F/G/K/M` и остается ли проблема уже как `O -> B`.
5. После этого связываем тот же hot-subset с текущим `final decision` run.
6. В конце фиксируем, нужен ли следующий пакет уже на границе `O vs B`.

In [3]:
# Конфигурация notebook.
COARSE_MODEL_RUN_DIR = REPO_ROOT / 'artifacts/models/gaia_id_coarse_classification__hist_gradient_boosting__2026_03_28_215003_509969'
FINAL_DECISION_RUN_DIR = REPO_ROOT / 'artifacts/decisions/hierarchical_final_decision_2026_03_29_111132_270743'
SUBSET_CONFIG = HotOBLikeSubsetConfig(teff_min_k=10000.0)
SOURCE_LIMIT = None
PREVIEW_ROWS = 20

if not COARSE_MODEL_RUN_DIR.exists():
    raise FileNotFoundError(COARSE_MODEL_RUN_DIR)
if not FINAL_DECISION_RUN_DIR.exists():
    raise FileNotFoundError(FINAL_DECISION_RUN_DIR)


In [4]:
# Загружаем review bundle для физически горячего подмножества.
bundle = load_hot_ob_like_subset_review_bundle_from_env(
    coarse_model_run_dir=COARSE_MODEL_RUN_DIR,
    final_decision_run_dir=FINAL_DECISION_RUN_DIR,
    config=SUBSET_CONFIG,
    source_limit=SOURCE_LIMIT,
    dotenv_path='.env',
)

display(build_hot_subset_membership_summary_frame(bundle))

,teff_min_k,n_rows_true_o_source,n_rows_hot_subset,share_hot_subset_in_source,n_rows_hot_pass,share_hot_pass_in_hot_subset,n_rows_hot_scored
0,10000.0,6372,3149,0.494193,1188,0.377263,1188


In [5]:
# Что делает quality-gate на физически горячем подмножестве.
display(build_hot_subset_quality_summary_frame(bundle.hot_subset_source_df))
display(build_hot_subset_quality_reason_frame(bundle.hot_subset_source_df, top_n=10))

,quality_state,n_rows,share
0,unknown,1961,0.622737
1,pass,1188,0.377263


,quality_reason,n_rows,share
0,missing_radius_flame,1754,0.557002
1,pass,1188,0.377263
2,high_ruwe,153,0.048587
3,low_parallax_snr,54,0.017148


In [6]:
# Что coarse-model предсказывает на hot pass-subset.
display(build_hot_subset_scored_prediction_frame(bundle.scored_pass_hot_subset_df))
display(build_hot_subset_predicted_physics_summary_frame(bundle.scored_pass_hot_subset_df))

,coarse_predicted_label,n_rows,share,mean_confidence,median_confidence,max_confidence
0,B,1188,1.0,0.999749,0.999846,0.999863


,coarse_predicted_label,n_rows,median_teff_gspphot,median_logg_gspphot,median_bp_rp,median_radius_flame
0,B,1188,15613.8865,3.6724,0.477677,4.778071


In [7]:
# Какой final outcome получает hot-subset в текущем full pipeline.
display(build_hot_subset_final_outcome_distribution_frame(bundle.final_hot_subset_df))
display(build_hot_subset_final_coarse_distribution_frame(bundle.final_hot_subset_df))
display(build_hot_subset_final_reason_frame(bundle.final_hot_subset_df, top_n=10))

,final_domain_state,n_rows,share
0,unknown,1961,0.622737
1,id,1100,0.349317
2,ood,88,0.027945


,final_coarse_class,n_rows,share
0,<NA>,2049,0.650683
1,B,1100,0.349317


,final_decision_reason,n_rows,share
0,quality_unknown,1961,0.622737
1,refinement_accepted,1100,0.349317
2,hard_ood,88,0.027945


In [8]:
# Самые уверенные hot-subset случаи, если coarse уводит объект не в `O/B`.
display(build_hot_subset_high_confidence_non_ob_preview_frame(bundle.scored_pass_hot_subset_df, top_n=PREVIEW_ROWS))

,source_id,coarse_predicted_label,coarse_probability_max,coarse_probability_margin,teff_gspphot,logg_gspphot,mh_gspphot,bp_rp,parallax,parallax_over_error,ruwe,radius_flame


## Опорные Источники

- [Gaia DR3 documentation index](https://gea.esac.esa.int/archive/documentation/GDR3/)
- [Gaia DR3 gaia_source semantics](https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_main_source_catalogue/ssec_dm_gaia_source.html)
- [Gaia DR3 astrophysical_parameters semantics](https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_astrophysical_parameter_tables/ssec_dm_astrophysical_parameters.html)
- [Astrophysical parameters associated to hot stars in Gaia DR3](https://www.aanda.org/articles/aa/full_html/2023/06/aa43709-22/aa43709-22.html)
- [NASA spectral classification overview](https://asd.gsfc.nasa.gov/archive/star_class/spectral_classification.html)
- [ESA star types overview](https://www.esa.int/Science_Exploration/Space_Science/Star_types)

## Следующие Шаги

- Если hot-subset по-прежнему полностью уходит в `B`, следующий пакет должен разбирать уже узкую границу `O vs B`, а не весь `O` source.
- Если после hot-cut начнут появляться стабильные `O`, тогда можно обсуждать targeted retrain/rebalance только на очищенном subset.
- Threshold и `priority` здесь не трогаем: этот notebook отвечает только за физически согласованный rare-tail review класса `O`.